In [1]:
from bell import *

from cirq_sic.wh import wh_povm
from cirq_sic.sics import load_sic_fiducial
from cirq_sic.utils import rand_ket

In [ ]:
d = 2
n_parties = 2
w = [[d,d] for i in range(n_parties)]
deterministic_behaviors, lambdas = construct_deterministic_behaviors(w, return_lambdas=True)

observables = [[rand_herm(d), rand_herm(d)] for i in range(n_parties)]
E = construct_povms_from_observables(observables)

ket = sum([kron(*[np.eye(d)[i]]*n_parties) for i in range(d)])/np.sqrt(d)
rho = np.outer(ket, ket.conj())

p = quantum_behavior_from_povms(E, rho); p

array([0.476, 0.024, 0.024, 0.476, 0.148, 0.352, 0.352, 0.148, 0.225,
       0.275, 0.275, 0.225, 0.199, 0.301, 0.301, 0.199])

In [87]:
L, problem = construct_hidden_variable_model(w, p, return_problem=True); L, problem.status

(array([0.03 , 0.184, 0.007, 0.005, 0.105, 0.158, 0.006, 0.005, 0.005,
        0.006, 0.158, 0.105, 0.005, 0.007, 0.184, 0.03 ]),
 'optimal')

In [88]:
bell_functional, classical_bound, problem = construct_bell_inequality(w, p, separation=1, dichotomous=False, return_problem=True)

print("LP objective:", problem.value)
print("max(D @ s - S):", np.max(deterministic_behaviors @ bell_functional - classical_bound))
print("p·s, S:", float(p @ bell_functional), float(classical_bound))

LP objective: -2.72260285533517e-11
max(D @ s - S): 1.559063586192921e-11
p·s, S: -3.863996836631707e-07 -3.8637245763461736e-07


In [89]:
d = 2
n_parties = 2

n_meas = 3
deterministic_behaviors = construct_deterministic_behaviors(w)

#w = [[d**2]*n_meas, [d**2]*n_meas]

#basis_povm = np.array([np.diag(np.eye(d)[i]) for i in range(d)])
#U = sc.stats.unitary_group.rvs(d)
#basis_povm2 = [U @ _ @ U.conj().T for _ in basis_povm]
#sic_povm = wh_povm(load_sic_fiducial(d))

#E = [[basis_povm, sic_povm], [transpose_povm(basis_povm), transpose_povm(sic_povm)]]
#E = [[basis_povm, sic_povm], [basis_povm, sic_povm]]
#E = [[sic_povm, transpose_povm(sic_povm)], [sic_povm, transpose_povm(sic_povm)]]
#E = [[basis_povm, sic_povm], [basis_povm2, transpose_povm(sic_povm)]]

#ket = sum([np.kron(np.eye(d)[i], np.eye(d)[i]) for i in range(d)])/np.sqrt(d)
#ket = np.kron(sc.stats.unitary_group.rvs(d), np.eye(d)) @ ket
#ket = rand_ket(d**2)

#E = [[wh_povm(rand_ket(d)), wh_povm(rand_ket(d))], [wh_povm(rand_ket(d)), wh_povm(rand_ket(d))]]

#ket = sum([np.kron(np.eye(d)[i], np.eye(d)[i]) for i in range(d)])/np.sqrt(d)
#rho = np.outer(ket, ket.conj())
#observables = [[rand_herm(d), rand_herm(d)], [rand_herm(d), rand_herm(d)]]
#observables = [[rand_dichotomous(), rand_dichotomous()], [rand_dichotomous(), rand_dichotomous()]]
#E = construct_povms_from_observables(observables)

sic_povm = wh_povm(load_sic_fiducial(d))
rotate = lambda povm, U: np.array([U @ e @ U.conj().T for e in povm])
E = [[rotate(sic_povm, rand_unitary(d)) for i in range(n_meas)], [rotate(sic_povm, rand_unitary(d)) for i in range(n_meas)]]
#E = [[wh_povm(load_sic_fiducial(d)), wh_povm(rand_ket(d)), wh_povm(rand_ket(d))], [wh_povm(load_sic_fiducial(d)), wh_povm(rand_ket(d)), wh_povm(rand_ket(d))]]
ket = sum([np.kron(np.eye(d)[i], np.eye(d)[i]) for i in range(d)])/np.sqrt(d)
rho = np.outer(ket, ket.conj())

p = quantum_behavior_from_povms(E, rho)
bell_functional, classical_bound, problem = construct_bell_inequality(w, p, dichotomous=False, return_problem=True)

print("LP objective:", problem.value)
print("max(D @ s - S):", np.max(deterministic_behaviors @ bell_functional - classical_bound))
print("p·s, S:", float(p @ bell_functional), float(classical_bound))

ValueError: Incompatible dimensions (16, 16) (144, 1)

In [7]:
L, problem = construct_hidden_variable_model(w, p, return_problem=True); L, problem.status

(None, 'infeasible')

In [8]:
np.allclose(L @ deterministic_behaviors, p)

ValueError: matmul: Input operand 0 does not have enough dimensions (has 0, gufunc core with signature (n?,k),(k,m?)->(n?,m?) requires 1)

In [19]:
bell_functional @ p >= classical_bound

np.True_

In [45]:
r = abs(np.random.randn(deterministic_behaviors.shape[0]))
r = r/np.sum(r)
classical_p = r @ deterministic_behaviors
classical_p @ bell_functional

np.float64(-1218.9431428203795)

In [46]:
reference_measurements = np.array([wh_povm(load_sic_fiducial(d)) for i in range(n_parties)])
reference_states = np.array([d*reference_measurements[i] for i in range(n_parties)])

T, T_meta = construct_T(rho, E, reference_measurements, reference_states, return_metadata=True)
phi = T_meta["phi"]
assert np.allclose(p, T @ phi)
quantumness_bound(bell_functional, classical_bound, T, phi)

np.float64(5.296135012357098e-05)

In [47]:
T_singular_values = np.linalg.svd(T, compute_uv=False)
Delta = bell_functional @ p - classical_bound
bound = Delta/(np.max(T_singular_values)*np.linalg.norm(bell_functional)); bound

np.float64(5.296135012289663e-05)

In [48]:
bell_functional @ p - classical_bound

np.float64(0.9999999976589606)

In [49]:
T_singular_values, np.linalg.norm(bell_functional)

(array([0.791, 0.403, 0.403, 0.403, 0.205, 0.205, 0.205, 0.152, 0.152,
        0.152, 0.152, 0.152, 0.152, 0.104, 0.077, 0.077, 0.077, 0.077,
        0.077, 0.077, 0.077, 0.077, 0.077, 0.077, 0.077, 0.077, 0.039,
        0.039, 0.039, 0.039, 0.039, 0.039, 0.029, 0.029, 0.029, 0.029,
        0.029, 0.029, 0.029, 0.029, 0.029, 0.029, 0.029, 0.029, 0.015,
        0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015, 0.015,
        0.015, 0.015, 0.006, 0.006, 0.006, 0.006, 0.006, 0.006, 0.006,
        0.006, 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
        0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   , 0.   ,
      

In [50]:
(np.max(T_singular_values)*np.linalg.norm(bell_functional))

np.float64(18881.693826506766)